In [4]:
import json
import os
import datetime
import requests
import findspark
findspark.init()

In [ ]:
# URLs for the API endpoints
BASE_URL = "https://pncp.gov.br/api/consulta/v1/"
PUBLISHED_TENDERS_URL = BASE_URL + "contratacoes/publicacao?dataInicial={start_data}&dataFinal={final_date}&codigoModalidadeContratacao={contracting_method_code}&pagina={page}"
OPEN_CALL_FOR_PROPOSALS_URL = BASE_URL + "contratacoes/proposta?dataFinal={date}&pagina={page}"
PRICE_REGISTRATION_MINUTES_URL = BASE_URL + "atas?dataInicial={start_data}&dataFinal={final_date}&pagina={page}"
SIGNED_CONTRACTS_TENDERS_URL = BASE_URL + "contratos?dataInicial={start_data}&dataFinal={final_date}&pagina={page}"

# Paths for saving the raw JSON responses
BRONZE_BASE_PATH = "./data/bronze/"
PUBLISHED_TENDERS_PATH = BRONZE_BASE_PATH + "tenders_published/published_tenders_{start_data}_{final_date}_page{page}.json"
OPEN_CALL_FOR_PROPOSALS_PATH = BRONZE_BASE_PATH + "tenders_open_procurement_processes/open_call_for_proposals_{date}_page{page}.json"
PRICE_REGISTRATION_MINUTES_PATH = BRONZE_BASE_PATH + "price_registration_minutes/price_registration_minutes_{start_data}_{final_date}_page{page}.json"
SIGNED_CONTRACTS_TENDERS_PATH = BRONZE_BASE_PATH + "signed_contracts_tenders/signed_contracts_tenders_{start_data}_{final_date}_page{page}.json"

# Mapping of contracting method codes to their descriptions
contracting_method_codes = {
    2: "Diálogo Competitivo",
    3: "Concurso",
    4: "Concorrência - Eletrônica",
    5: "Concorrência - Presencial",
    6: "Pregão - Eletrônico",
    7: "Pregão - Presencial",
    10: "Manifestação de Interesse",
    11: "Pré-qualificação",
    12: "Credenciamento"
}

In [ ]:
# Verify if the directory exists, if not, create it
os.makedirs(BRONZE_BASE_PATH, exist_ok=True)
os.makedirs(PUBLISHED_TENDERS_PATH, exist_ok=True)
os.makedirs(OPEN_CALL_FOR_PROPOSALS_PATH, exist_ok=True)
os.makedirs(PRICE_REGISTRATION_MINUTES_PATH, exist_ok=True)
os.makedirs(SIGNED_CONTRACTS_TENDERS_PATH, exist_ok=True)

In [9]:
today = datetime.date.today()
start_date = today if today.weekday() == 0 else today - datetime.timedelta(days=today.weekday()) #Monday of the current week
end_date = today if today.weekday() == 4 else today + datetime.timedelta(days=4 - today.weekday()) #Friday of the current week

In [ ]:
def fetch_published_tenders(start_data, final_date, contracting_method_codes):
    for contracting_method_code in contracting_method_codes:
        page = 1
        remaining_pages = 1
        while remaining_pages > 0:
            try:
                response = requests.get(
                    PUBLISHED_TENDERS_URL.format(
                        start_data=start_data,
                        final_date=final_date,
                        contracting_method_code=contracting_method_code,
                        page=page
                    )
                )
                response_json = response.json()
                remaining_pages = response_json['paginasRestantes']
                with open(PUBLISHED_TENDERS_PATH.format(start_data=start_data, final_date=final_date, page=page), 'w', encoding='utf-8') as f:
                    json.dump(response_json, f, ensure_ascii=False, indent=4)
            except Exception as e:
                print(f"Error fetching published tenders for contracting method code {contracting_method_code} on page {page}: {e}")
                break
            else:
                page += 1

fetch_published_tenders(start_date.strftime("%Y%m%d"), end_date.strftime("%Y%m%d"), contracting_method_codes.keys())

In [12]:
def fetch_open_call_for_proposals(date):
    page = 1
    remaining_pages = 1
    while remaining_pages > 0:
        try:
            response = requests.get(
                OPEN_CALL_FOR_PROPOSALS_URL.format(date=date, page=page)
            )
            response_json = response.json()
            remaining_pages = response_json['paginasRestantes']
            with open(OPEN_CALL_FOR_PROPOSALS_PATH.format(date=date, page=page), 'w', encoding='utf-8') as f:
                json.dump(response_json, f, ensure_ascii=False, indent=4)
        except Exception as e:
            print(f"Error fetching open call for proposals on page {page}: {e}")
            break
        else:
            page += 1

fetch_open_call_for_proposals(end_date.strftime("%Y%m%d"))

Error fetching open call for proposals on page 3: HTTPSConnectionPool(host='pncp.gov.br', port=443): Max retries exceeded with url: /api/consulta/v1/contratacoes/proposta?dataFinal=20261002&pagina=3 (Caused by ConnectTimeoutError(<HTTPSConnection(host='pncp.gov.br', port=443) at 0x7187f4545f70>, 'Connection to pncp.gov.br timed out. (connect timeout=None)'))


In [13]:
def fetch_price_registration_minutes(start_data, final_date):
    page = 1
    remaining_pages = 1
    while remaining_pages > 0:
        try:
            response = requests.get(
                PRICE_REGISTRATION_MINUTES_URL.format(start_data=start_data, final_date=final_date, page=page)
            )
            response_json = response.json()
            remaining_pages = response_json['paginasRestantes']
            with open(PRICE_REGISTRATION_MINUTES_PATH.format(start_data=start_data, final_date=final_date, page=page), 'w', encoding='utf-8') as f:
                json.dump(response_json, f, ensure_ascii=False, indent=4)
        except Exception as e:
            print(f"Error fetching price registration minutes on page {page}: {e}")
            break
        else:
            page += 1

fetch_price_registration_minutes(start_date.strftime("%Y%m%d"), end_date.strftime("%Y%m%d"))

Error fetching price registration minutes on page 3: HTTPSConnectionPool(host='pncp.gov.br', port=443): Max retries exceeded with url: /api/consulta/v1/atas?dataInicial=20260928&dataFinal=20261002&pagina=3 (Caused by ConnectTimeoutError(<HTTPSConnection(host='pncp.gov.br', port=443) at 0x7187f4539640>, 'Connection to pncp.gov.br timed out. (connect timeout=None)'))


In [14]:
def fetch_signed_contracts_tenders(start_data, final_date):
    page = 1
    remaining_pages = 1
    while remaining_pages > 0:
        try:
            response = requests.get(
                SIGNED_CONTRACTS_TENDERS_URL.format(start_data=start_data, final_date=final_date, page=page)
            )
            response_json = response.json()
            remaining_pages = response_json['paginasRestantes']
            with open(SIGNED_CONTRACTS_TENDERS_PATH.format(start_data=start_data, final_date=final_date, page=page), 'w', encoding='utf-8') as f:
                json.dump(response_json, f, ensure_ascii=False, indent=4)
        except Exception as e:
            print(f"Error fetching signed contracts tenders on page {page}: {e}")
            break
        else:
            page += 1

fetch_signed_contracts_tenders(start_date.strftime("%Y%m%d"), end_date.strftime("%Y%m%d"))

Error fetching signed contracts tenders on page 18: HTTPSConnectionPool(host='pncp.gov.br', port=443): Max retries exceeded with url: /api/consulta/v1/contratos?dataInicial=20260928&dataFinal=20261002&pagina=18 (Caused by ConnectTimeoutError(<HTTPSConnection(host='pncp.gov.br', port=443) at 0x7187f453c350>, 'Connection to pncp.gov.br timed out. (connect timeout=None)'))
